In [21]:
# Select target dataset for this notebook
TARGET_DATASET = "Dataset102_MNI"  # options: Dataset101_MSD, Dataset102_MNI, Dataset103_ADNI, Dataset105_COBRA
DATASET_ID = int(TARGET_DATASET.split('_')[0].replace('Dataset', ''))
DATASET_CODE = TARGET_DATASET.split('_')[1]
import os
os.environ['TARGET_DATASET'] = TARGET_DATASET
os.environ['DATASET_ID'] = str(DATASET_ID)
os.environ['DATASET_CODE'] = DATASET_CODE


In [22]:
%mkdir /content

mkdir: cannot create directory ‘/content’: File exists


In [23]:

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
except:
    from google.colab import userdata
    wandb_api_key = userdata.get('WANDB_API_KEY')


In [24]:
# Write the .netrc file
netrc_content = f"""machine api.wandb.ai
  login user
  password {wandb_api_key}
"""

with open("/root/.netrc", "w") as f:
    f.write(netrc_content)

# Make sure permissions are correct
import os
os.chmod("/root/.netrc", 0o600)

In [25]:
%cd /content
!git clone https://github.com/BouncyButton/hippopotamus
!pip install wget
!mkdir datasets
%cd /content/datasets/
import os
script_map = {
    "Dataset101_MSD": "/content/hippopotamus/datasets/Dataset101_MSD/create_msd_dataset.py",
    "Dataset102_MNI": "/content/hippopotamus/datasets/Dataset102_MNI/create_mni_dataset.py",
    "Dataset103_ADNI": "/content/hippopotamus/datasets/Dataset103_ADNI/create_adni_dataset.py",
    "Dataset105_COBRA": "/content/hippopotamus/datasets/Dataset105_COBRA/create_cobra_dataset.py",
}
script = script_map.get(TARGET_DATASET)
if script is None:
    raise ValueError(f"Unknown TARGET_DATASET: {TARGET_DATASET}")
os.system(f"python {script}")


/content
fatal: destination path 'hippopotamus' already exists and is not an empty directory.
mkdir: cannot create directory ‘datasets’: File exists
/content/datasets


0

In [26]:
!pip install nnunetv2

  Using cached argparse-1.4.0-py2.py3-none-any.whl.metadata (2.8 kB)
Using cached argparse-1.4.0-py2.py3-none-any.whl (23 kB)


In [27]:
%cd ..

/content


In [28]:
import os

currdir = os.getcwd()

# set env
os.environ['nnUNet_raw'] = currdir + '/datasets'
os.environ['nnUNet_preprocessed'] = currdir + '/preprocessed'
os.environ['nnUNet_results'] = currdir + '/results'

In [29]:
!cp hippopotamus/datasets/$TARGET_DATASET/dataset.json $nnUNet_raw/$TARGET_DATASET/dataset.json

In [30]:
!nnUNetv2_plan_and_preprocess -d $DATASET_ID --verify_dataset_integrity

Fingerprint extraction...
Dataset102_MNI
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer
100% 50/50 [00:23<00:00,  2.09it/s]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Dropping 3d_lowres config because the image size difference to 3d_fullres is too small. 3d_fullres: [48. 56. 40.], 3d_lowres: [48, 56, 40]
2D U-Net configuration:
{'data_identifier': 'nnUNetPlans_2d', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 120, 'patch_size': (np.int64(56), np.

In [31]:
%cd /content

/content


In [32]:
%cd preprocessed/$TARGET_DATASET


/content/preprocessed/Dataset102_MNI


In [33]:
!ls -la

total 60
drwxr-xr-x 5 root root  4096 Feb  9 16:24 .
drwxr-xr-x 4 root root  4096 Feb  9 16:23 ..
-rw-r--r-- 1 root root  7434 Feb  9 16:23 dataset_fingerprint.json
-rw-r--r-- 1 root root   184 Feb  9 16:23 dataset.json
drwxr-xr-x 2 root root  4096 Feb  9 16:24 gt_segmentations
drwxr-xr-x 2 root root 12288 Feb  9 16:23 nnUNetPlans_2d
drwxr-xr-x 2 root root 12288 Feb  9 16:24 nnUNetPlans_3d_fullres
-rw-r--r-- 1 root root  9093 Feb  9 16:23 nnUNetPlans.json


## running a fake run to generate splits_final.json

In [ ]:
!timeout 15 nnUNetv2_train $DATASET_ID 3d_fullres 0


In [ ]:
%cd /content/preprocessed/$TARGET_DATASET


In [ ]:
!ls -a

In [ ]:
!python /content/hippopotamus/datasets/fix_nnunet_splits.py -i /content/preprocessed/$TARGET_DATASET/splits_final.json -o /content/preprocessed/$TARGET_DATASET/splits_final.json -d $DATASET_CODE


now "2026-01-27 09:49:22.937830: Using splits from existing split file: /content/preprocessed/Dataset102_MNI/splits_final.json"

In [ ]:
!nnUNetv2_train $DATASET_ID 3d_fullres 1 -tr nnUNetTrainer_250epochs
!python /content/hippopotamus/baselines/save_nnunet_run.py --model-path /content/results/$TARGET_DATASET/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_1 --fold 1 --dataset $DATASET_CODE
!nnUNetv2_train $DATASET_ID 3d_fullres 2 -tr nnUNetTrainer_250epochs
!python /content/hippopotamus/baselines/save_nnunet_run.py --model-path /content/results/$TARGET_DATASET/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_2 --fold 2 --dataset $DATASET_CODE
!nnUNetv2_train $DATASET_ID 3d_fullres 3 -tr nnUNetTrainer_250epochs
!python /content/hippopotamus/baselines/save_nnunet_run.py --model-path /content/results/$TARGET_DATASET/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_3 --fold 3 --dataset $DATASET_CODE
!nnUNetv2_train $DATASET_ID 3d_fullres 4 -tr nnUNetTrainer_250epochs
!python /content/hippopotamus/baselines/save_nnunet_run.py --model-path /content/results/$TARGET_DATASET/nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres/fold_4 --fold 4 --dataset $DATASET_CODE



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was pr